# 08 · Search, deduplication and energies

The normal `import rxembed as rx` API takes a SMILES, an `.xyz` path, a Mol or a metal Isomer and
returns a chainable `Ensemble` result.

- `mc()` runs an openconf torsional search, `prune()` dedups, both in place so they chain
- `lowest`, `representatives`, `align`, `score`, `optimize` return a new ensemble and leave this one alone
- `measure`, `cluster`, `landscape` change nothing

Do not compare surrogate force-field energies for a metal or charged TS. The calculator
ladder is `ff -> gfnff -> gfn2 -> gxtb`. `score()` is a single point, `optimize()` relaxes and holds any
frozen core. Energies are kcal/mol and only relative values mean anything. Set `$XTB_EXE`, or install `xtb` at `~/bin/xtb`.


In [ ]:
import rxembed as rx
import xyzrender

rx.set_verbose("INFO")
IBU = "CC(C)Cc1ccc(C(C)C(=O)O)cc1"  # ibuprofen; several real rotamers

## The chain

`embed()` returns one RDKit Mol holding many conformers. After pruning, `.mol` contains the kept conformers
and `.discarded` records the merged IDs. `prune(by=...)` offers `rmsd` (the safe default), `moi` (a cheap prefilter), `descriptor`,
`energy`, and `none`; methods can be cascaded, for example `by=["moi", "rmsd"]`. Use
`representatives()` instead when the goal is one conformer per distinct binding mode.


In [ ]:
ens = rx.embed("OC(=O)CCCCc1ccccc1").mc().minimize().prune(by="rmsd")
print(len(ens), "kept,", len(ens.discarded), "merged away")
labels = ens.cluster()
print("cluster populations (-1 is noise):", {int(k): int(sum(labels == k)) for k in set(labels)})
print(repr(ens))

## The geometry gate

`geom_check.check` reports broken conjugation, bad H positions,
non-bonded clashes, stretched bonds, a moved frozen core.


In [ ]:
rep = ens.check()[ens.ids[0]]
print("clean" if rep.ok() else f"violations: {rep.violations[:4]}")

## The pool: embed → search → prune → keep the lowest few

Build a diverse, deduplicated pool on the cheap FF, gate it, then spend the calculator on the best candidates. `wrap(mol, ids)` turns selected conformers of an existing Mol back into an Ensemble without re-embedding them.

In [ ]:
# stereo="free": ibuprofen's undefined α-stereocentre would split this into an EnsembleSet of two
# enantiomers (identical energies): the tier ladder below is about energy, not stereochemistry.
pool = rx.embed(IBU, stereo="free").mc().minimize().prune()
pool.filter("geometry")
cand = pool.lowest(6)  # the clean force-field minima to re-rank with a real energy
print(f"geometry.check: {pool.n} clean conformers -> re-ranking the lowest {len(cand)}")

## Energy methods: ff → gfnff → gfn2 → g-xTB

`score()` runs a single point without moving atoms; `relative()` gives ΔE from the minimum. The example
compares all four methods on the same conformers. Each `score()` returns a new ensemble.

In [ ]:
print("ΔE (kcal/mol) across the tiers on the same 6 candidates:")
print(f"  ff   : {sorted(round(e, 2) for e in cand.relative().values())}")  # the FF pool energies
print(f"  gfnff: {sorted(round(e, 2) for e in cand.score('gfnff').relative().values())}")
print(f"  gfn2 : {sorted(round(e, 2) for e in cand.score('gfn2').relative().values())}")
scored = cand.score("gxtb")  # keep the g-xTB single points (reused in the non-clobber check below)
print(f"  gxtb : {sorted(round(e, 2) for e in scored.relative().values())}")
print("lowest by g-xTB:", scored.lowest(1).ids)

## GFN-FF pool ranking, then g-xTB

GFN-FF includes dispersion and hydrogen-bond terms. Use it to rank the pool before running g-xTB on the
lowest few conformers.

In [ ]:
ranked = pool.score("gfnff").prune()  # NCI-aware pool re-rank
print("GFN-FF ΔE:", sorted(round(e, 1) for e in ranked.relative().values())[:6])
final = ranked.lowest(2).optimize("gxtb", level="loose")  # g-xTB opt, the best few only
for report in final.check().values():
    report.assert_ok()
print("g-xTB opt ΔE:", sorted(round(e, 2) for e in final.relative().values()))
xyzrender.render(final.mol, no_hy=True)

## Failure and mutation contracts

`score('gxtb')` and `optimize('gxtb')` raise if the calculator produces no energy. g-xTB solvent uses the
GFN2 correction `E_gxtb(gas) + E_gfn2(solv) - E_gfn2(gas)`. Both methods return new ensembles.

In [ ]:
print("input pool untouched:", len(pool), "conformers on the FF path")
print("scored / optimised are independent ensembles:", len(scored), "/", len(final))
# g-xTB in solvent: no ALPB, so a GFN2 thermodynamic-cycle correction; runs, never a raise or silent gas-phase
wet = cand.lowest(2).score("gxtb", solvent="water")
print(f"g-xTB/water on {len(wet)} conf: ΔE {sorted(round(e, 2) for e in wet.relative().values())} (solvent-corrected)")